<a href="https://colab.research.google.com/github/astikarganesh/25-26-4093-Ganesh-AI-B-DS/blob/main/WEEK-10/Social_Network_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

# 37) Write Code to Implement Social Network Analysis

This notebook implements comprehensive social network analysis including:
1. Graph Creation and Visualization
2. Network Metrics (Centrality Measures)
3. PageRank Algorithm
4. Community Detection
5. Shortest Path Analysis

## 1. Import Required Libraries

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from itertools import combinations
import pandas as pd

print("Libraries imported successfully!")

## 2. Create Social Network Graph

In [ ]:
# Create an undirected graph representing a social network
G = nx.Graph()

# Add nodes (people in the network)
nodes = ['Alice', 'Bob', 'Charlie', 'David', 'Eve', 'Frank', 'Grace', 'Henry']
G.add_nodes_from(nodes)

# Add edges (friendships/connections)
edges = [
    ('Alice', 'Bob'),
    ('Alice', 'Charlie'),
    ('Bob', 'Charlie'),
    ('Bob', 'David'),
    ('Charlie', 'David'),
    ('David', 'Eve'),
    ('Eve', 'Frank'),
    ('Frank', 'Grace'),
    ('Grace', 'Henry'),
    ('Frank', 'Henry'),
    ('Alice', 'Grace')
]
G.add_edges_from(edges)

print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")
print(f"Network density: {nx.density(G):.3f}")

## 3. Visualize the Network

In [ ]:
# Create visualization
plt.figure(figsize=(12, 8))

# Use spring layout for better visualization
pos = nx.spring_layout(G, k=2, iterations=50, seed=42)

# Draw the network
nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=1500)
nx.draw_networkx_edges(G, pos, width=2, alpha=0.6)
nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')

plt.title("Social Network Graph", fontsize=16, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

print("Network visualization complete!")

## 4. Calculate Centrality Measures

In [ ]:
# Calculate different centrality measures

# Degree Centrality: Number of direct connections
degree_cent = nx.degree_centrality(G)
print("=" * 50)
print("DEGREE CENTRALITY (Number of connections)")
print("=" * 50)
for node, value in sorted(degree_cent.items(), key=lambda x: x[1], reverse=True):
    print(f"{node}: {value:.4f}")

# Closeness Centrality: Average distance to all other nodes
closeness_cent = nx.closeness_centrality(G)
print("\n" + "=" * 50)
print("CLOSENESS CENTRALITY (Proximity to all nodes)")
print("=" * 50)
for node, value in sorted(closeness_cent.items(), key=lambda x: x[1], reverse=True):
    print(f"{node}: {value:.4f}")

# Betweenness Centrality: How often a node lies on shortest paths
betweenness_cent = nx.betweenness_centrality(G)
print("\n" + "=" * 50)
print("BETWEENNESS CENTRALITY (Bridge importance)")
print("=" * 50)
for node, value in sorted(betweenness_cent.items(), key=lambda x: x[1], reverse=True):
    print(f"{node}: {value:.4f}")

# Eigenvector Centrality: Importance based on connections to important nodes
eigenvector_cent = nx.eigenvector_centrality(G, max_iter=1000)
print("\n" + "=" * 50)
print("EIGENVECTOR CENTRALITY (Quality of connections)")
print("=" * 50)
for node, value in sorted(eigenvector_cent.items(), key=lambda x: x[1], reverse=True):
    print(f"{node}: {value:.4f}")

## 5. PageRank Algorithm (Main Implementation)

In [ ]:
# Manual PageRank Implementation
def pagerank_manual(graph, d=0.85, max_iterations=100, tolerance=1e-6):
    """
    Calculate PageRank for each node in the graph.
    
    Parameters:
    - graph: NetworkX graph object
    - d: damping factor (usually 0.85)
    - max_iterations: maximum iterations for convergence
    - tolerance: convergence threshold
    
    Returns:
    - Dictionary with PageRank scores for each node
    """
    
    nodes = list(graph.nodes())
    n = len(nodes)
    
    # Initialize PageRank values to 1/n
    pagerank = {node: 1/n for node in nodes}
    
    # Iterations to calculate PageRank
    for iteration in range(max_iterations):
        prev_pagerank = pagerank.copy()
        
        for node in nodes:
            # Get incoming links
            incoming_nodes = list(graph.predecessors(node))
            if not incoming_nodes:
                incoming_nodes = list(graph.neighbors(node))
            
            rank = (1 - d) / n  # Random surfer component
            
            for incoming_node in incoming_nodes:
                out_degree = graph.degree(incoming_node)
                if out_degree > 0:
                    rank += d * (prev_pagerank[incoming_node] / out_degree)
            
            pagerank[node] = rank
        
        # Check for convergence
        diff = sum(abs(pagerank[node] - prev_pagerank[node]) for node in nodes)
        if diff < tolerance:
            print(f"Converged at iteration {iteration}")
            break
    
    return pagerank

# Calculate PageRank using manual implementation
manual_pagerank = pagerank_manual(G)
print("\n" + "=" * 50)
print("PAGERANK ALGORITHM (Manual Implementation)")
print("=" * 50)
for node, value in sorted(manual_pagerank.items(), key=lambda x: x[1], reverse=True):
    print(f"{node}: {value:.6f}")

In [ ]:
# NetworkX Built-in PageRank
nx_pagerank = nx.pagerank(G, alpha=0.85)
print("\n" + "=" * 50)
print("PAGERANK ALGORITHM (NetworkX Implementation)")
print("=" * 50)
for node, value in sorted(nx_pagerank.items(), key=lambda x: x[1], reverse=True):
    print(f"{node}: {value:.6f}")

## 6. Visualize PageRank Scores

In [ ]:
# Create a bar plot for PageRank scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: PageRank scores as bar chart
pr_values = sorted(nx_pagerank.items(), key=lambda x: x[1], reverse=True)
nodes_pr = [item[0] for item in pr_values]
scores_pr = [item[1] for item in pr_values]

axes[0].barh(nodes_pr, scores_pr, color='skyblue', edgecolor='navy')
axes[0].set_xlabel('PageRank Score', fontsize=12)
axes[0].set_title('PageRank Scores by Node', fontsize=14, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

# Plot 2: Network with node sizes proportional to PageRank
pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
node_sizes = [nx_pagerank[node] * 5000 for node in G.nodes()]

nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='lightcoral', ax=axes[1])
nx.draw_networkx_edges(G, pos, width=2, alpha=0.6, ax=axes[1])
nx.draw_networkx_labels(G, pos, font_size=9, font_weight='bold', ax=axes[1])

axes[1].set_title('Network with Node Sizes = PageRank Scores', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 7. Shortest Path Analysis

In [ ]:
# Calculate shortest paths
print("=" * 50)
print("SHORTEST PATH ANALYSIS")
print("=" * 50)

# Shortest path between two nodes
start_node = 'Alice'
end_node = 'Henry'
shortest_path = nx.shortest_path(G, start_node, end_node)
path_length = nx.shortest_path_length(G, start_node, end_node)

print(f"\nShortest path from {start_node} to {end_node}:")
print(f"Path: {' -> '.join(shortest_path)}")
print(f"Path Length: {path_length}")

# Average shortest path length
avg_shortest_path = nx.average_shortest_path_length(G)
print(f"\nAverage Shortest Path Length: {avg_shortest_path:.4f}")

# Diameter of the network
diameter = nx.diameter(G)
print(f"Network Diameter: {diameter}")

## 8. Community Detection

In [ ]:
from networkx.algorithms import community

print("=" * 50)
print("COMMUNITY DETECTION")
print("=" * 50)

# Greedy modularity communities
communities_list = list(community.greedy_modularity_communities(G))

print(f"\nNumber of communities detected: {len(communities_list)}")
for i, comm in enumerate(communities_list):
    print(f"Community {i+1}: {comm}")

# Calculate modularity
modularity = community.modularity(G, communities_list)
print(f"\nModularity Score: {modularity:.4f}")

## 9. Network Statistics Summary

In [ ]:
# Comprehensive network statistics
print("\n" + "=" * 50)
print("NETWORK STATISTICS SUMMARY")
print("=" * 50)

stats_data = {
    'Number of Nodes': G.number_of_nodes(),
    'Number of Edges': G.number_of_edges(),
    'Network Density': f"{nx.density(G):.4f}",
    'Average Clustering Coefficient': f"{nx.average_clustering(G):.4f}",
    'Network Diameter': diameter,
    'Average Shortest Path': f"{avg_shortest_path:.4f}",
    'Is Connected': nx.is_connected(G),
    'Number of Connected Components': nx.number_connected_components(G),
    'Number of Communities': len(communities_list),
    'Modularity Score': f"{modularity:.4f}"
}

for key, value in stats_data.items():
    print(f"{key}: {value}")

## 10. Visualization with Node Colors by Community

In [ ]:
# Create color map for communities
color_map = {}
colors = ['lightblue', 'lightcoral', 'lightgreen', 'lightyellow', 'lightpink']

for i, comm in enumerate(communities_list):
    for node in comm:
        color_map[node] = colors[i % len(colors)]

# Create visualization
plt.figure(figsize=(12, 8))

pos = nx.spring_layout(G, k=2, iterations=50, seed=42)

# Draw nodes with community colors
node_colors = [color_map[node] for node in G.nodes()]
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1500, edgecolors='black', linewidths=2)

# Draw edges
nx.draw_networkx_edges(G, pos, width=2, alpha=0.6)

# Draw labels
nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')

plt.title("Social Network with Detected Communities", fontsize=16, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

print("Network visualization with communities complete!")

## 11. Comparative Analysis Table

In [ ]:
# Create a comprehensive comparison table
comparison_data = {
    'Node': list(G.nodes()),
    'Degree': [G.degree(node) for node in G.nodes()],
    'Degree Centrality': [f"{degree_cent[node]:.4f}" for node in G.nodes()],
    'Closeness Centrality': [f"{closeness_cent[node]:.4f}" for node in G.nodes()],
    'Betweenness Centrality': [f"{betweenness_cent[node]:.4f}" for node in G.nodes()],
    'PageRank': [f"{nx_pagerank[node]:.6f}" for node in G.nodes()],
    'Eigenvector Centrality': [f"{eigenvector_cent[node]:.4f}" for node in G.nodes()]
}

df = pd.DataFrame(comparison_data)
print("\n" + "=" * 120)
print("COMPREHENSIVE CENTRALITY MEASURES TABLE")
print("=" * 120)
print(df.to_string(index=False))
print("=" * 120)

## Summary

**Key Social Network Analysis Concepts Implemented:**

1. **Degree Centrality**: Measures direct connections; higher value = more connections
2. **Closeness Centrality**: Measures how close a node is to all other nodes
3. **Betweenness Centrality**: Measures how often a node appears on shortest paths (bridge importance)
4. **Eigenvector Centrality**: Measures importance based on connections to other important nodes
5. **PageRank Algorithm**: Ranks nodes based on the importance of their incoming connections
6. **Community Detection**: Identifies groups of densely connected nodes
7. **Shortest Path Analysis**: Finds the shortest connections between nodes
8. **Network Statistics**: Density, clustering coefficient, diameter, and connectivity metrics

These metrics help identify influential nodes, community structures, and network characteristics in social networks.